## 07 — Replay and storage tiers
Assert event-ID uniqueness through views, distinguish physical duplicates, inspect codec/storage evidence.

In [ ]:
import clickhouse_connect, os
client = clickhouse_connect.get_client(
    host=os.environ.get('CLICKHOUSE_HOST', 'localhost'),
    port=int(os.environ.get('CLICKHOUSE_PORT', '8123')),
    username=os.environ.get('CLICKHOUSE_USER', 'default'),
    password=os.environ.get('CLICKHOUSE_PASSWORD', ''),
    database=os.environ.get('CLICKHOUSE_DATABASE', 'market'),
)
print('clickhouse', client.server_version)


In [ ]:
dups = client.query("SELECT count() FROM (SELECT _record_run_id, _record_writer_id, _record_sequence, _record_row_index, count() AS n FROM trades FINAL GROUP BY 1,2,3,4 HAVING n > 1)").result_rows[0][0]
assert dups == 0, f'{dups} duplicate identities'
print('event identity uniqueness OK')

In [ ]:
parts = client.query("SELECT table, count(), sum(rows) FROM system.parts WHERE database='market' AND active GROUP BY table").result_rows
print(parts)

In [ ]:
physical = client.query("SELECT count() FROM (SELECT _record_run_id, _record_writer_id, _record_sequence, count() AS n FROM trades GROUP BY 1,2,3 HAVING n > 1)").result_rows[0][0]
print('physical duplicates pre-merge (may exist):', physical)